# Crea tu propio ChatGPT — RAG desde cero

Construiremos:
`documentos → chunking → overlap → embeddings → similarity search → Top-K → contexto → LLM`

## Instalación

```bash
uv add jupyter ipykernel openai numpy pandas scikit-learn tiktoken python-dotenv
```

Opcional:
```bash
uv add faiss-cpu
```

In [ ]:
# Instalación de dependencias: instala en este kernel los paquetes que falten (si ya están, no hace nada)
import sys, shutil, subprocess, importlib, importlib.util

PAQUETES = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "tiktoken": "tiktoken",
            "openai": "openai", "dotenv": "python-dotenv", "matplotlib": "matplotlib"}

def _faltantes():
    importlib.invalidate_caches()
    return [pip for mod, pip in PAQUETES.items() if importlib.util.find_spec(mod) is None]

faltan = _faltantes()
if faltan:
    print("Instalando:", ", ".join(faltan), "→", sys.executable, "(puede tardar 1-2 minutos)")
    intentos = []
    if shutil.which("uv"):
        intentos.append(["uv", "pip", "install", "--python", sys.executable, *faltan])
    intentos.append([sys.executable, "-m", "pip", "install", "--quiet", "--retries", "1", "--timeout", "20", *faltan])
    for cmd in intentos:
        if cmd[0] == sys.executable and subprocess.run(cmd[:3] + ["--version"], capture_output=True).returncode != 0:
            subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], capture_output=True)
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode == 0 and not _faltantes():
            break
        print(f"⚠️ Falló el intento con {cmd[0]}; probando otra vía...\n", (r.stderr or r.stdout)[-800:])
    faltan = _faltantes()
    if faltan:
        raise ModuleNotFoundError(
            f"No se pudieron instalar: {', '.join(faltan)} en {sys.executable}\n"
            "Revisa tu conexión a internet. Solución: en la terminal, dentro del repositorio, ejecuta `uv sync`; luego en VS Code elige "
            "Select Kernel → Python Environments → .venv (la del repositorio) y reinicia el kernel."
        )
print("✅ Dependencias listas:", sys.executable)

In [ ]:
import os
import numpy as np
from dotenv import load_dotenv
load_dotenv()
print("API key configurada:", bool(os.getenv("OPENAI_API_KEY")))

## IA tradicional vs IA generativa

IA tradicional puede resolver clasificación, regresión y predicción.
IA generativa produce contenido nuevo condicionado por instrucciones y contexto.

RAG no re-entrena el modelo: recupera información externa y la proporciona al modelo durante la consulta.

## Tokens

Los modelos procesan texto mediante tokens. Una palabra no necesariamente equivale a un token.

Los tokens afectan contexto, costos y límites.

In [ ]:
import tiktoken

text = "La inteligencia artificial generativa puede responder usando documentos privados."
enc = tiktoken.get_encoding("cl100k_base")
tokens = enc.encode(text)

print("Texto:", text)
print("Tokens:", tokens)
print("Cantidad:", len(tokens))

## Documento de ejemplo

In [ ]:
document = """La empresa ofrece servicios de soporte técnico para clientes empresariales.
El horario de atención es de lunes a viernes de 9:00 a 18:00 horas.
Los clientes pueden abrir tickets mediante el portal de soporte.
Los tickets críticos deben incluir una descripción del problema y evidencia.
El tiempo objetivo de primera respuesta para incidentes críticos es de una hora.
El equipo de soporte puede solicitar información adicional antes de escalar un incidente.
"""
print(document)

## Chunking

Un chunk es una porción del documento que será indexada y recuperada.

No existe un tamaño universal. Debemos experimentar con documentos y preguntas reales.

In [ ]:
def chunk_text(text, chunk_size=300, overlap=50):
    if overlap >= chunk_size:
        raise ValueError("overlap debe ser menor que chunk_size")
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(document, 300, 50)
for i, chunk in enumerate(chunks):
    print(f"--- CHUNK {i} ---\n{chunk}\n")

## ¿Cómo estimar cantidad de chunks?

Aproximación:
`chunks ≈ longitud_documento / (chunk_size - overlap)`

Ejemplo: 10,000 caracteres, chunk 1,000, overlap 200:
10,000 / 800 ≈ 12.5

Pero caracteres no son tokens. En producción conviene considerar tokens y, sobre todo, estructura/semántica del documento.

In [ ]:
def estimate_chunks(text_length, chunk_size, overlap):
    return int(np.ceil(text_length / (chunk_size - overlap)))

print(estimate_chunks(10000, 1000, 200))

## Overlap

El overlap mantiene parte del contexto entre chunks.

Demasiado poco puede romper contexto; demasiado puede producir duplicación, almacenamiento y resultados redundantes.

In [ ]:
for overlap in [0, 50, 100]:
    c = chunk_text(document, 250, overlap)
    print("overlap =", overlap, "chunks =", len(c))

## Embeddings

Un embedding representa texto como un vector numérico. Textos semánticamente relacionados pueden quedar próximos en el espacio vectorial.

No interpretes cada dimensión como una palabra concreta: es una representación distribuida.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(chunks)
print("Matriz TF-IDF:", X.shape)

## Similarity search

Primero usamos TF-IDF para entender el mecanismo. Luego usaremos embeddings reales.

In [ ]:
query = "¿Cuál es el horario del equipo de soporte?"
q = vectorizer.transform([query])
scores = cosine_similarity(q, X)[0]

for idx in np.argsort(scores)[::-1][:3]:
    print(f"score={scores[idx]:.3f} | chunk={idx}")
    print(chunks[idx], "\n")

## Embeddings reales con OpenAI

Esta sección requiere `OPENAI_API_KEY`.

El nombre del modelo puede cambiar; verifica la documentación actual del proveedor antes de producción.

In [ ]:
from openai import OpenAI
client = OpenAI()

def get_embeddings(texts, model="text-embedding-3-small"):
    response = client.embeddings.create(model=model, input=texts)
    return np.array([item.embedding for item in response.data])

real_embeddings = get_embeddings(chunks)
print(real_embeddings.shape)

## Cosine similarity

`cos(A,B) = (A · B) / (||A|| ||B||)`

Valores más cercanos a 1 indican orientación vectorial más similar.

In [ ]:
def cosine(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom else 0.0

query_embedding = get_embeddings([query])[0]
scores = [cosine(query_embedding, emb) for emb in real_embeddings]

for idx in np.argsort(scores)[::-1][:3]:
    print(f"score={scores[idx]:.4f} | chunk={idx}")
    print(chunks[idx], "\n")

## Top-K retrieval

Top-K recupera los K chunks con mayor similarity.

K pequeño: menos ruido/contexto.
K grande: más cobertura, pero potencialmente más ruido y costo.

In [ ]:
def retrieve(query, chunks, embeddings, k=3):
    q_emb = get_embeddings([query])[0]
    scores = np.array([cosine(q_emb, emb) for emb in embeddings])
    top = np.argsort(scores)[::-1][:k]
    return [{"chunk_id": int(i), "score": float(scores[i]), "text": chunks[i]} for i in top]

results = retrieve(query, chunks, real_embeddings, 3)
for r in results:
    print(r)

## Context assembly

In [ ]:
context = "\n\n".join(
    f"[Chunk {r['chunk_id']}]\n{r['text']}" for r in results
)
print(context)

## Generation

Prompt conceptual:

```text
Responde usando únicamente el contexto.
CONTEXTO: ...
PREGUNTA: ...
Si no está en el contexto, dilo.
```

Esto ayuda a hacer grounding, aunque no garantiza respuestas perfectas.

In [ ]:
prompt = f'''
Responde usando únicamente el contexto proporcionado.
Si no existe información suficiente, dilo claramente.

CONTEXTO:
{context}

PREGUNTA:
{query}
'''
print(prompt)

## LLM

In [ ]:
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": "Responde de forma precisa y no inventes información."},
        {"role": "user", "content": prompt}
    ],
    temperature=0
)
print(response.choices[0].message.content)

## RAG completo

In [ ]:
def rag(question, chunks, embeddings, k=3):
    retrieved = retrieve(question, chunks, embeddings, k)
    context = "\n\n".join(
        f"[Chunk {r['chunk_id']}]\n{r['text']}" for r in retrieved
    )
    prompt = f'''
Responde usando únicamente el contexto.
Si no existe información suficiente, dilo claramente.

CONTEXTO:
{context}

PREGUNTA:
{question}
'''
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": "Responde de forma precisa y grounded."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    return {"answer": response.choices[0].message.content, "retrieved": retrieved}

result = rag("¿Cuál es el horario del equipo de soporte?", chunks, real_embeddings, 3)
print(result["answer"])

## Pregunta fuera del conocimiento

Prueba una pregunta que no exista en el documento. Observa qué recupera el sistema y cómo responde el modelo.

In [ ]:
result = rag("¿Cuál es el precio de la licencia empresarial?", chunks, real_embeddings, 3)
print(result["answer"])

# Laboratorio final

Modifica:
- chunk_size
- overlap
- k
- preguntas
- prompt

Compara resultados.

Preguntas:
1. ¿Qué sucede con chunks pequeños?
2. ¿Qué sucede con chunks grandes?
3. ¿Qué ocurre con overlap 0?
4. ¿Qué ocurre con overlap muy alto?
5. ¿Qué ocurre al aumentar K?
6. ¿Cómo medir retrieval quality?

# Arquitectura final

```text
DOCUMENTOS
    ↓
CHUNKING + OVERLAP + METADATA
    ↓
EMBEDDINGS
    ↓
VECTOR INDEX
    ↓
QUERY → EMBEDDING
    ↓
SIMILARITY SEARCH
    ↓
TOP-K
    ↓
CONTEXT
    ↓
LLM
    ↓
ANSWER
```

Frameworks como LangChain/LangGraph abstraen estas piezas, pero ahora ya entendemos qué ocurre debajo.

## Ideas para producción

- Document loaders
- Chunking semántico
- Metadata filtering
- Qdrant / pgvector / Pinecone
- Hybrid search
- Reranking
- Query rewriting
- Citations
- Observabilidad
- Evaluación automática
- Guardrails
- Caching